In [ ]:
# =============================================================================
# Extract pixel-level feature tables from GEE assets → CSV
#
# Time split:
#   Training   : 2010–2020
#   Validation : 2021–2022
#   Test       : 2023–2024  (locked — not used in this script)
# =============================================================================

import ee
import time
import os

# ── 0. Auth ───────────────────────────────────────────────────────────────────
KEY_FILE        = 'ru-thesis-2026-66238f2c1197.json'
SERVICE_ACCOUNT = 'ru-thesis-gee-api@ru-thesis-2026.iam.gserviceaccount.com'
PROJECT         = 'ru-thesis-2026'

# credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, KEY_FILE)
# ee.Initialize(credentials, project=PROJECT)

# ── Re-initialise with personal account for Drive export ─────────────────────
# The service account cannot write to Drive.

ee.Authenticate()
ee.Initialize(project=PROJECT)


# ── 1. Configuration ──────────────────────────────────────────────────────────
STATIC_FOLDER  = f"projects/{PROJECT}/assets/Ontario_Static_2010-2024"
DYNAMIC_FOLDER = f"projects/{PROJECT}/assets/Ontario_Monthly_2010-2024"

# ── All features from monthly dynamic stack ───────────────────────────────────
# Band name prefixes — the function appends _YYYYMMDD automatically
DYNAMIC_FEATURES = [
    'temp_max',         # daily maximum temperature (°C)
    'rh_min',           # daily minimum relative humidity (%)
    'vpd_mean',         # daily mean vapour pressure deficit (kPa)
    'wind_speed_mean',  # daily mean wind speed (m/s)
    'precip_sum',       # daily total precipitation (mm)
    'solar_rad_mj',     # daily net solar radiation (MJ/m²)
    'rh_7d',            # 7-day rolling mean RH (%)
    'temp_7d',          # 7-day rolling mean temperature (°C)
    'precip_30d',       # 30-day accumulated precipitation (mm)
    'recovery_value',   # post-fire recovery proxy [0–1]
]

# ── All features from static asset ────────────────────────────────────────────
# Exact band names — no date suffix
STATIC_FEATURES = [
    'elevation_m',        # terrain elevation (m)
    'slope_deg',          # terrain slope (degrees)
    'land_cover',         # NALCMS 2020 land cover class
    'pop_density',        # WorldPop population density
    'road_proximity_m',   # distance to nearest road (m)
]

# ── Target variables ──────────────────────────────────────────────────────────
TARGETS = ['target_ignition', 'fire_cause']

# ── province_id is included for stratification but not as a model feature ─────

STRATIFICATION = ['province_id']

# Combined feature list used for model training (excludes targets and strat)
FEATURES = DYNAMIC_FEATURES + STATIC_FEATURES



# Time split
TRAIN_YEARS = list(range(2010, 2021))   # 2010–2020
VAL_YEARS   = [2021, 2022]
TEST_YEARS  = [2023, 2024]              # locked — not extracted here

DRIVE_FOLDER = 'GEE_Ontario_ML'

ontario_fc   = (ee.FeatureCollection("FAO/GAUL/2015/level1")
                .filter(ee.Filter.eq('ADM0_NAME', 'Canada'))
                .filter(ee.Filter.eq('ADM1_NAME', 'Ontario')))
ontario_geom = ontario_fc.geometry().simplify(4500)

era5_sample  = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").first()
GRID_PROJ    = era5_sample.select(0).projection()
GRID_SCALE   = GRID_PROJ.nominalScale()

# ── 2. GEE extraction: flatten monthly stacks to pixel-level tables ───────────
#
# Strategy:
#   For each year, load the static asset (one image) and each monthly
#   dynamic asset (12 images). For each day in the monthly image, extract
#   the relevant bands, join to static, and sample all Ontario pixels.
#   Export as one CSV per year to Google Drive.
#

def get_month_str(year, month):
    return f"{year}{month:02d}"


def extract_year_to_drive(year, split_label):
    print(f"\n  Extracting {year} ({split_label})...")

    static_img   = ee.Image(f"{STATIC_FOLDER}/Static_{year}")
    static_bands = static_img.select(STATIC_FEATURES + STRATIFICATION)

    # Skip already submitted tasks
    try:
        existing_tasks = {
            t['description']
            for t in ee.data.listOperations()
            if t.get('metadata', {}).get('state') in
               ('RUNNING', 'READY', 'COMPLETED', 'SUCCEEDED')
        }
    except Exception:
        existing_tasks = set()

    for month in range(1, 13):
        month_str   = f"{year}{month:02d}"
        description = f'ML_{split_label}_{month_str}'

        if description in existing_tasks:
            print(f"    {month_str}: already submitted — skipping")
            continue

        monthly_id = f"{DYNAMIC_FOLDER}/Month_{month_str}"

        try:
            monthly_img = ee.Image(monthly_id)
            all_bands   = monthly_img.bandNames().getInfo()

            dates_in_month = sorted(set(
                b.replace('temp_max_', '')
                for b in all_bands
                if b.startswith('temp_max_')
            ))

            if not dates_in_month:
                print(f"    {month_str}: no dates — skipping")
                continue

            all_day_features = []
            for d_str in dates_in_month:
                bands_to_add = []
                for feat in DYNAMIC_FEATURES:
                    bn = f'{feat}_{d_str}'
                    if bn in all_bands:
                        bands_to_add.append(
                            monthly_img.select(bn).rename(feat))
                for tgt in TARGETS:
                    bn = f'{tgt}_{d_str}'
                    if bn in all_bands:
                        bands_to_add.append(
                            monthly_img.select(bn).rename(tgt))

                if len(bands_to_add) < len(DYNAMIC_FEATURES) + len(TARGETS):
                    continue

                day_stack = bands_to_add[0]
                for b in bands_to_add[1:]:
                    day_stack = day_stack.addBands(b)

                # Add static bands and lon/lat as numeric bands
                # lon/lat stored as regular columns — no geometry needed
                day_stack = (day_stack
                             .addBands(static_bands)
                             .addBands(ee.Image.pixelLonLat()))

                # sample() with geometries=False — toDrive CSV accepts null geometry
                day_fc = (day_stack
                          .sample(
                              region     = ontario_geom,
                              scale      = GRID_SCALE,
                              projection = GRID_PROJ,
                              geometries = False)   # no geometry — avoids all geometry errors
                          .map(lambda f: f.set('date', d_str)))

                all_day_features.append(day_fc)

            if not all_day_features:
                print(f"    {month_str}: no valid days — skipping")
                continue

            merged = all_day_features[0]
            for fc in all_day_features[1:]:
                merged = merged.merge(fc)

            task = ee.batch.Export.table.toDrive(
                collection     = merged,
                description    = description,
                folder         = DRIVE_FOLDER,
                fileNamePrefix = f'ML_{split_label}_{month_str}',
                fileFormat     = 'CSV',
                selectors      = (['date', 'longitude', 'latitude']
                                  + DYNAMIC_FEATURES
                                  + STATIC_FEATURES
                                  + STRATIFICATION
                                  + TARGETS)
            )
            
            task.start()
            print(f"    {month_str}: submitted "
                  f"({len(dates_in_month)} days, "
                  f"~{len(dates_in_month) * 12606:,} expected rows)")

        except Exception as e:
            print(f"    {month_str}: ERROR — {e}")


# ── Submit one month directly as a test ────────


print("Submitting April 2010 as a test export...")
extract_year_to_drive(2010, 'train')
print("\nMonitor: https://code.earthengine.google.com/tasks")
print("When complete, check the CSV in Google Drive:")
print("  - Row count should be ~12,606 × 30 days = ~378,180 rows")
print("  - Columns: date, longitude, latitude, temp_max, precip_30d,")
print("             land_cover, target_ignition, fire_cause")
print("  - target_ignition should have some 1s (fire pixels)")

Submitting April 2010 as a test export...

  Extracting 2010 (train)...
    201001: submitted (31 days, ~390,786 expected rows)
    201002: submitted (28 days, ~352,968 expected rows)
    201003: submitted (31 days, ~390,786 expected rows)
    201004: submitted (30 days, ~378,180 expected rows)
    201005: submitted (31 days, ~390,786 expected rows)
    201006: submitted (30 days, ~378,180 expected rows)
    201007: submitted (31 days, ~390,786 expected rows)
    201008: submitted (31 days, ~390,786 expected rows)
    201009: submitted (30 days, ~378,180 expected rows)
    201010: submitted (31 days, ~390,786 expected rows)
    201011: submitted (30 days, ~378,180 expected rows)
    201012: submitted (31 days, ~390,786 expected rows)

Monitor: https://code.earthengine.google.com/tasks
When complete, check the CSV in Google Drive:
  - Row count should be ~12,606 × 30 days = ~378,180 rows
  - Columns: date, longitude, latitude, temp_max, precip_30d,
             land_cover, target_igniti

In [3]:
# ── Verify all band prefixes exist in a sample monthly asset ──────────────────
test_img  = ee.Image(f"{DYNAMIC_FOLDER}/Month_201001")
all_bands = test_img.bandNames().getInfo()

print("Checking dynamic feature band prefixes in Month_201001:")
for prefix in DYNAMIC_FEATURES:
    matches = [b for b in all_bands if b.startswith(prefix + '_')]
    status  = 'OK' if matches else '<-- MISSING'
    print(f"  {prefix:<22} {len(matches):>3} bands  {status}")

print("\nChecking static feature bands in Static_2010:")
static_test = ee.Image(f"{STATIC_FOLDER}/Static_2010")
static_bands_present = static_test.bandNames().getInfo()
for feat in STATIC_FEATURES + STRATIFICATION:
    status = 'OK' if feat in static_bands_present else '<-- MISSING'
    print(f"  {feat:<22} {status}")

Checking dynamic feature band prefixes in Month_201001:
  temp_max                31 bands  OK
  rh_min                  31 bands  OK
  vpd_mean                31 bands  OK
  wind_speed_mean         31 bands  OK
  precip_sum              31 bands  OK
  solar_rad_mj            31 bands  OK
  rh_7d                   31 bands  OK
  temp_7d                 31 bands  OK
  precip_30d              31 bands  OK
  recovery_value          31 bands  OK

Checking static feature bands in Static_2010:
  elevation_m            OK
  slope_deg              OK
  land_cover             OK
  pop_density            OK
  road_proximity_m       OK
  province_id            OK


In [ ]:
# ── Full extraction — 2011 to 2024 ───────────────────────────────────────────
# 2010 training data already exported 
# Test years (2023-2024) extracted separately with 'test' label

BATCH_PAUSE = 15   # seconds between years to avoid overwhelming the API

# Training years: 2011–2020
TRAIN_YEARS_REMAINING = list(range(2011, 2021))

# Validation years: 2021–2022
VAL_YEARS = [2021, 2022]

# Test years: 2023–2024 (same extraction function, different label)
TEST_YEARS_TO_EXTRACT = [2023, 2024]

print("=" * 55)
print(f"Full extraction plan:")
print(f"  Training   (train) : {TRAIN_YEARS_REMAINING}")
print(f"  Validation (val)   : {VAL_YEARS}")
print(f"  Test       (test)  : {TEST_YEARS_TO_EXTRACT}")
print(f"  Total tasks        : "
      f"{(len(TRAIN_YEARS_REMAINING) + len(VAL_YEARS) + len(TEST_YEARS_TO_EXTRACT)) * 12}"
      f" monthly exports")
print("=" * 55)

# ── Training years 2011–2020 ──────────────────────────────────────────────────
print("\nSubmitting training years (2011–2020)...")
for year in TRAIN_YEARS_REMAINING:
    extract_year_to_drive(year, 'train')
    time.sleep(BATCH_PAUSE)

# ── Validation years 2021–2022 ────────────────────────────────────────────────
print("\nSubmitting validation years (2021–2022)...")
for year in VAL_YEARS:
    extract_year_to_drive(year, 'val')
    time.sleep(BATCH_PAUSE)

# ── Test years 2023–2024 ──────────────────────────────────────────────────────
# Extracted now but NOT used until final model evaluation.

print("\nSubmitting test years (2023–2024)...")
for year in TEST_YEARS_TO_EXTRACT:
    extract_year_to_drive(year, 'test')
    time.sleep(BATCH_PAUSE)

print("\n" + "=" * 55)
print("All extraction tasks submitted.")
print(f"Monitor: https://code.earthengine.google.com/tasks")
print(f"\nExpected CSVs in Google Drive folder '{DRIVE_FOLDER}':")
print(f"  ML_train_201101.csv ... ML_train_202012.csv  (120 files)")
print(f"  ML_val_202101.csv   ... ML_val_202212.csv    ( 24 files)")
print(f"  ML_test_202301.csv  ... ML_test_202412.csv   ( 24 files)")
print(f"  ML_train_201001.csv ... ML_train_201012.csv  ( 12 files, already done)")
print(f"  Total: 180 CSV files")
print(f"\nDo NOT load ML_test_*.csv files until final model evaluation.")